# Camada Silver

In [0]:
# ============================================================
# CAMADA SILVER - TRATAMENTO E PADRONIZAÇÃO DOS DADOS
# Projeto: Avaliação de Alfabetização - INEP
# Objetivo:
# - Ler dados da camada Bronze
# - Remover duplicidades
# - Filtrar registros inválidos
# - Padronizar textos
# - Converter tipos de dados
# - Salvar tabelas tratadas em Delta na camada Silver
# ============================================================

In [0]:
# ============================================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    current_timestamp,
    lit
)

In [0]:
# ============================================================
# 2. CONFIGURAÇÕES GERAIS DO PROJETO
# ============================================================

CATALOG = "workspace"

SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"

# Cria o schema Silver caso ainda não exista
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}")

DataFrame[]

In [0]:
# ============================================================
# 3. FUNÇÃO PARA CONVERTER TIPOS DE COLUNAS
# ============================================================

def converter_colunas(df: DataFrame, colunas_tipos: dict) -> DataFrame:
    """
    Converte colunas para os tipos definidos no dicionário.

    Exemplo:
    {
        "ano": "int",
        "taxa_alfabetizacao": "double"
    }
    """

    for coluna, tipo in colunas_tipos.items():
        if coluna in df.columns:
            df = df.withColumn(coluna, col(coluna).cast(tipo))

    return df

In [0]:
# ============================================================
# 4. FUNÇÃO PARA PADRONIZAR COLUNAS DE TEXTO
# ============================================================

def limpar_colunas_texto(
    df: DataFrame,
    colunas_upper: list = None,
    colunas_trim: list = None
) -> DataFrame:
    """
    Aplica limpeza básica em colunas textuais.

    trim  -> remove espaços no começo e no final
    upper -> transforma texto em maiúsculo
    """

    colunas_upper = colunas_upper or []
    colunas_trim = colunas_trim or []

    for coluna in colunas_upper:
        if coluna in df.columns:
            df = df.withColumn(coluna, upper(trim(col(coluna))))

    for coluna in colunas_trim:
        if coluna in df.columns:
            df = df.withColumn(coluna, trim(col(coluna)))

    return df

In [0]:
# ============================================================
# 5. FUNÇÃO PARA SALVAR TABELAS NA SILVER
# ============================================================

def salvar_silver(df: DataFrame, nome_tabela: str) -> None:
    """
    Salva o DataFrame tratado como tabela Delta na camada Silver.
    """

    tabela_destino = f"{CATALOG}.{SCHEMA_SILVER}.{nome_tabela}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_destino)
    )

    print(f"Tabela salva na Silver: {tabela_destino}")
    print(f"Total de registros: {df.count()}")

In [0]:
# ============================================================
# 6. COLUNAS PADRÃO DAS TABELAS DE META
# ============================================================

colunas_metas = {
    "taxa_alfabetizacao": "double",
    "meta_alfabetizacao_2024": "double",
    "meta_alfabetizacao_2025": "double",
    "meta_alfabetizacao_2026": "double",
    "meta_alfabetizacao_2027": "double",
    "meta_alfabetizacao_2028": "double",
    "meta_alfabetizacao_2029": "double",
    "meta_alfabetizacao_2030": "double",
    "percentual_participacao": "double"
}

In [0]:
# ============================================================
# 7. CONFIGURAÇÃO DAS TABELAS SILVER
# ============================================================

config_silver = {
    "br_inep_avaliacao_alfabetizacao_municipio": {
        "colunas_obrigatorias": [
            "ano",
            "id_municipio"
        ],
        "tipos": {
            "ano": "int",
            "id_municipio": "string",
            "serie": "int",
            "rede": "int",
            "taxa_alfabetizacao": "double",
            "media_portugues": "double"
        }
    },

    "br_inep_avaliacao_alfabetizacao_uf": {
        "colunas_obrigatorias": [
            "ano",
            "sigla_uf"
        ],
        "upper": [
            "sigla_uf"
        ],
        "tipos": {
            "ano": "int",
            "serie": "int",
            "rede": "int",
            "taxa_alfabetizacao": "double",
            "media_portugues": "double"
        }
    },

    "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil": {
        "colunas_obrigatorias": [
            "ano",
            "rede"
        ],
        "trim": [
            "rede"
        ],
        "tipos": {
            "ano": "int",
            **colunas_metas
        }
    },

    "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf": {
        "colunas_obrigatorias": [
            "ano",
            "sigla_uf",
            "rede"
        ],
        "upper": [
            "sigla_uf"
        ],
        "trim": [
            "rede"
        ],
        "tipos": {
            "ano": "int",
            **colunas_metas
        }
    },

    "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio": {
        "colunas_obrigatorias": [
            "ano",
            "id_municipio",
            "rede"
        ],
        "trim": [
            "rede"
        ],
        "tipos": {
            "ano": "int",
            "id_municipio": "string",
            **colunas_metas,
            "nivel_alfabetizacao": "double"
        }
    }
}

In [0]:
# ============================================================
# 8. PIPELINE PRINCIPAL DA CAMADA SILVER
# ============================================================

for nome_tabela, config in config_silver.items():

    print("=" * 80)
    print(f"Iniciando processamento Silver: {nome_tabela}")

    # --------------------------------------------------------
    # 8.1 Leitura da tabela Bronze
    # --------------------------------------------------------

    df = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{nome_tabela}")

    # --------------------------------------------------------
    # 8.2 Remoção de registros duplicados
    # --------------------------------------------------------

    df = df.dropDuplicates()

    # --------------------------------------------------------
    # 8.3 Filtro de colunas obrigatórias
    # Remove registros sem chaves ou métricas essenciais
    # --------------------------------------------------------

    for coluna in config.get("colunas_obrigatorias", []):
        if coluna in df.columns:
            df = df.filter(col(coluna).isNotNull())

    # --------------------------------------------------------
    # 8.4 Padronização de colunas textuais
    # --------------------------------------------------------

    df = limpar_colunas_texto(
        df=df,
        colunas_upper=config.get("upper", []),
        colunas_trim=config.get("trim", [])
    )

    # --------------------------------------------------------
    # 8.5 Conversão dos tipos de dados
    # --------------------------------------------------------

    df = converter_colunas(
        df=df,
        colunas_tipos=config.get("tipos", {})
    )

    # --------------------------------------------------------
    # 8.6 Inclusão de metadados técnicos da camada Silver
    # --------------------------------------------------------

    df = (
        df
        .withColumn("_data_processamento_silver", current_timestamp())
        .withColumn("_camada", lit("silver"))
    )

    # --------------------------------------------------------
    # 8.7 Escrita da tabela tratada na Silver
    # --------------------------------------------------------

    salvar_silver(df, nome_tabela)

    # --------------------------------------------------------
    # 8.8 Visualização rápida para validação
    # --------------------------------------------------------

    display(df.limit(5))

    print(f"Processamento finalizado: {nome_tabela}")

Iniciando processamento Silver: br_inep_avaliacao_alfabetizacao_municipio
Tabela salva na Silver: workspace.silver.br_inep_avaliacao_alfabetizacao_municipio
Total de registros: 23995


ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_id_linha,_data_ingestao,_arquivo_origem,_caminho_arquivo,_camada,_data_processamento_silver
2023,1100031,2,3,69.1,767.8763,null,null,null,null,null,null,null,null,null,0,2026-06-24T13:45:34.448Z,br_inep_avaliacao_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_municipio.csv,silver,2026-06-24T16:42:06.915Z
2023,1100072,2,3,58.2,747.8918,null,null,null,null,null,null,null,null,null,1,2026-06-24T13:45:34.448Z,br_inep_avaliacao_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_municipio.csv,silver,2026-06-24T16:42:06.915Z
2023,1100189,2,5,69.73,762.4062,null,null,null,null,null,null,null,null,null,2,2026-06-24T13:45:34.448Z,br_inep_avaliacao_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_municipio.csv,silver,2026-06-24T16:42:06.915Z
2023,1101609,2,3,50.7,745.6802,null,null,null,null,null,null,null,null,null,3,2026-06-24T13:45:34.448Z,br_inep_avaliacao_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_municipio.csv,silver,2026-06-24T16:42:06.915Z
2023,1101807,2,3,55.69,752.3724,null,null,null,null,null,null,null,null,null,4,2026-06-24T13:45:34.448Z,br_inep_avaliacao_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_municipio.csv,silver,2026-06-24T16:42:06.915Z


Processamento finalizado: br_inep_avaliacao_alfabetizacao_municipio
Iniciando processamento Silver: br_inep_avaliacao_alfabetizacao_uf
Tabela salva na Silver: workspace.silver.br_inep_avaliacao_alfabetizacao_uf
Total de registros: 145


ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_id_linha,_data_ingestao,_arquivo_origem,_caminho_arquivo,_camada,_data_processamento_silver
2023,AM,2,3,49.2,733.6637,null,null,null,null,null,null,null,null,null,0,2026-06-24T13:45:39.528Z,br_inep_avaliacao_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_uf.csv,silver,2026-06-24T16:42:16.108Z
2023,PB,2,2,55.23,744.8152,null,null,null,null,null,null,null,null,null,1,2026-06-24T13:45:39.528Z,br_inep_avaliacao_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_uf.csv,silver,2026-06-24T16:42:16.108Z
2023,PR,2,5,73.12,757.2146,null,null,null,null,null,null,null,null,null,2,2026-06-24T13:45:39.528Z,br_inep_avaliacao_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_uf.csv,silver,2026-06-24T16:42:16.108Z
2023,AP,2,3,41.87,732.7858,null,null,null,null,null,null,null,null,null,3,2026-06-24T13:45:39.528Z,br_inep_avaliacao_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_uf.csv,silver,2026-06-24T16:42:16.108Z
2023,PE,2,5,58.95,747.4522,null,null,null,null,null,null,null,null,null,4,2026-06-24T13:45:39.528Z,br_inep_avaliacao_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_uf.csv,silver,2026-06-24T16:42:16.108Z


Processamento finalizado: br_inep_avaliacao_alfabetizacao_uf
Iniciando processamento Silver: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil
Tabela salva na Silver: workspace.silver.br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil
Total de registros: 3


...

Processamento finalizado: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil
Iniciando processamento Silver: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf
Tabela salva na Silver: workspace.silver.br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf
Total de registros: 54


ano,sigla_uf,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,_id_linha,_data_ingestao,_arquivo_origem,_caminho_arquivo,_camada,_data_processamento_silver
2024,RR,Pública,null,null,null,null,null,null,null,null,null,0,2026-06-24T13:45:29.640Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,silver,2026-06-24T16:42:32.373Z
2023,RR,Pública,null,null,null,null,null,null,null,null,null,1,2026-06-24T13:45:29.640Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,silver,2026-06-24T16:42:32.373Z
2023,AC,Pública,null,null,56.9,62.2,67.3,72.0,76.2,80.0,null,2,2026-06-24T13:45:29.640Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,silver,2026-06-24T16:42:32.373Z
2024,AC,Pública,51.38,null,56.9,62.2,67.3,72.0,76.2,80.0,80.87,3,2026-06-24T13:45:29.640Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,silver,2026-06-24T16:42:32.373Z
2023,DF,Pública,null,null,63.1,67.0,70.6,74.0,77.1,80.0,null,4,2026-06-24T13:45:29.640Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv,silver,2026-06-24T16:42:32.373Z


Processamento finalizado: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf
Iniciando processamento Silver: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio
Tabela salva na Silver: workspace.silver.br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio
Total de registros: 10704


ano,id_municipio,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao,_id_linha,_data_ingestao,_arquivo_origem,_caminho_arquivo,_camada,_data_processamento_silver
2023,4301750,Municipal,null,null,14.05,23.65,37.0,52.68,67.85,80.0,null,null,0,2026-06-24T13:45:24.615Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,silver,2026-06-24T16:42:41.496Z
2024,4301750,Municipal,4.4,null,14.05,23.65,37.0,52.68,67.85,80.0,0.0,92.59,1,2026-06-24T13:45:24.615Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,silver,2026-06-24T16:42:41.496Z
2024,2406908,Municipal,42.86,7.94,14.05,23.65,37.0,52.68,67.85,80.0,1.0,84.0,2,2026-06-24T13:45:24.615Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,silver,2026-06-24T16:42:41.496Z
2023,2406908,Municipal,4.4,7.94,14.05,23.65,37.0,52.68,67.85,80.0,0.0,82.14,3,2026-06-24T13:45:24.615Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,silver,2026-06-24T16:42:41.496Z
2023,1718501,Municipal,4.6,8.25,14.48,24.16,37.49,53.03,68.0,80.0,0.0,95.65,4,2026-06-24T13:45:24.615Z,br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,dbfs:/Volumes/workspace/default/tech_challenge_alfabetizacao/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv,silver,2026-06-24T16:42:41.496Z


Processamento finalizado: br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio
